# To Run Spectral Ratio Illumination Demo on Google Colab

Omar Elmady 

Wednesday, Dec 11 

CS 7180 

## Setup Steps:
1. **Enable GPU Runtime**: Runtime → Change runtime type → Hardware accelerator → **GPU** → Save
2. **Update model link** in Step 2 if you have it from your professor
3. **Run all cells in order** (Runtime → Run all)

## What this notebook does:
- Checks GPU availability
- Downloads model from Google Drive (if link provided)
- Clones your GitHub repository
- Installs all dependencies (PyTorch with GPU/CPU support)
- Runs validation checks
- Processes images with your algorithms
- Downloads results as a .tar.gz file

## Step 1: Check GPU Availability

In [ ]:
# Check if GPU is available
!nvidia-smi || echo "⚠️ No GPU detected - will use CPU (slower but works)"

## Step 2: Clone Repository and Download Model

### 2a. Clone Repository

First, clone the GitHub repository to get all the code and data.

In [ ]:
import os

# Remove if already exists from previous runs
if os.path.exists('/content/Spectral_Ratio_Illumination_Demo'):
    !rm -rf /content/Spectral_Ratio_Illumination_Demo

# Clone the repository
print("📥 Cloning repository from GitHub...")
!git clone https://github.com/oelmady/Spectral_Ratio_Illumination_Demo.git /content/Spectral_Ratio_Illumination_Demo

# Change to the project directory
%cd /content/Spectral_Ratio_Illumination_Demo

print("\n✓ Repository cloned successfully!")
print("\n📁 Repository contents:")
!ls -lh

### 2b. Download Model from Google Drive

To extract file ID from a Drive link like `https://drive.google.com/file/d/1ABC123XYZ/view?usp=sharing`, copy the `1ABC123XYZ` part. If you need to change it, update `MODEL_DRIVE_ID` below.  

The model will download directly into the repository's `model/` folder.

In [ ]:
import os

# ============================================
# CONFIGURATION: Google Drive file ID for model
# ============================================
MODEL_DRIVE_ID = "1h2fVtLQJpgLl4_C3MLA_VDuqlJTcAqf6"
# ============================================

if MODEL_DRIVE_ID:
    print("📥 Downloading model from Google Drive...")
    
    # Install gdown for Drive downloads
    !pip install -q gdown
    
    import gdown
    
    # Download directly into the repo's model/ directory
    model_path = '/content/Spectral_Ratio_Illumination_Demo/model/UNET_run_x10_01_last_model.pth'
    url = f'https://drive.google.com/uc?id={MODEL_DRIVE_ID}'
    
    try:
        gdown.download(url, model_path, quiet=False)
        
        # Verify download
        if os.path.exists(model_path):
            size_mb = os.path.getsize(model_path) / (1024 * 1024)
            if size_mb > 100:  # Should be ~528MB
                print(f"\n✓ Model downloaded successfully: {size_mb:.1f} MB")
                print(f"   Location: {model_path}")
            else:
                print(f"\n⚠️ Model file seems too small ({size_mb:.1f} MB)")
                print("   Check if the Drive link allows public access")
        else:
            print("\n❌ Model download failed")
            print("   Make sure the file is shared with 'Anyone with the link'")
    except Exception as e:
        print(f"\n❌ Error downloading model: {e}")
        print("   Double-check the file ID and sharing permissions")
else:
    print("⚠️ No model file ID provided")
    print("   Will run baseline-only experiments (no neural ISD prediction)")

# Show model directory contents
print("\n📁 Model directory:")
!ls -lh /content/Spectral_Ratio_Illumination_Demo/model/


## Step 3: Install Dependencies

This cell installs all required Python packages:
- PyTorch (GPU version if available, otherwise CPU)
- OpenCV (full version with GUI support)
- NumPy, Matplotlib, and other dependencies

In [ ]:
import subprocess
import sys

print("📦 Installing dependencies...\n")

# Upgrade pip
print("1️⃣ Upgrading pip...")
!pip install --quiet --upgrade pip setuptools wheel

# Install opencv and scientific computing libraries
print("\n2️⃣ Installing OpenCV, NumPy, Matplotlib, scikit-image...")
!pip install --quiet opencv-python numpy matplotlib scikit-image scipy

# Install PyTorch with GPU support if available
print("\n3️⃣ Installing PyTorch...")
try:
    subprocess.run(["nvidia-smi"], check=True, capture_output=True)
    print("   GPU detected: Installing CUDA-enabled PyTorch (cu118)")
    !pip install --quiet torch torchvision --index-url https://download.pytorch.org/whl/cu118
except:
    print("   No GPU: Installing CPU-only PyTorch")
    !pip install --quiet torch torchvision --index-url https://download.pytorch.org/whl/cpu

# Install any remaining requirements
print("\n4️⃣ Installing remaining requirements...")
!pip install --quiet -r requirements.txt 2>/dev/null || true

print("\n✓ All dependencies installed successfully!")

# Verify installations
print("\n📋 Checking installed versions:")
import torch
import cv2
import numpy as np
from skimage import __version__ as skimage_version
print(f"   PyTorch: {torch.__version__}")
print(f"   CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
print(f"   OpenCV: {cv2.__version__}")
print(f"   NumPy: {np.__version__}")
print(f"   scikit-image: {skimage_version}")

## Step 4: Run Preflight Check

This validates that everything is set up correctly.

In [ ]:
%cd /content/Spectral_Ratio_Illumination_Demo
!python preflight_check.py

## Step 5: Run Experiments

culatioThis cell processes all images in `data/images/` with four algorithms.
- Neural ISD prediction
- SR-constrained Retinex
- Baseline Retinex (for comparison)
- SR-based color correction

In [ ]:
import os
import sys

%cd /content/Spectral_Ratio_Illumination_Demo

# Set PYTHONPATH environment variable so subprocess can find modules
os.environ['PYTHONPATH'] = '/content/Spectral_Ratio_Illumination_Demo'

model_path = '/content/Spectral_Ratio_Illumination_Demo/model/UNET_run_x10_01_last_model.pth'

if os.path.exists(model_path) and os.path.getsize(model_path) > 1000000:  # > 1MB
    print("🚀 Running FULL experiment (with neural ISD model)...")
    print("   This includes: model inference + SR-Retinex + baseline Retinex + color correction")
    print("   Plus: Quality metrics (SSIM + color constancy)\n")
    !python scripts/run_batch.py \
        --use-model \
        --retinex \
        --baseline-retinex \
        --sr-correct \
        --compute-metrics \
        --iterations 5 \
        --sigma 15 \
        --distance 1.0
else:
    print("🚀 Running BASELINE experiment (no model)...")
    print("   This includes: baseline Retinex + SR-constrained Retinex (using annotated maps)\n")
    print("   ⚠️ Note: Since no model is available, this will use pre-existing SR maps")
    print("   from data/sr_maps/ if available, or skip SR-constrained processing.\n")
    !python scripts/run_batch.py \
        --baseline-retinex \
        --retinex \
        --compute-metrics \
        --iterations 5 \
        --sigma 15

print("\n✓ Processing complete! Check results/ directory for images and quality_metrics.json")

## Step 6: View Sample Results

Display a few output images to verify processing worked correctly.

In [ ]:
import os
import glob
from IPython.display import Image, display
import matplotlib.pyplot as plt

# Find PNG outputs in results directory
png_files = glob.glob('/content/Spectral_Ratio_Illumination_Demo/results/*.png')

if png_files:
    print(f"📸 Found {len(png_files)} output images. Showing first 3:\n")
    for img_path in png_files[:3]:
        print(f"   {os.path.basename(img_path)}")
        display(Image(filename=img_path, width=600))
        print()
else:
    print("⚠️ No PNG outputs found in results/")
    print("   Check if processing completed successfully above.")

## Step 7: Package and Download Results

This creates a `.tar.gz` archive of all results and downloads it to your local machine.

In [ ]:
import os
from google.colab import files

%cd /content/Spectral_Ratio_Illumination_Demo

# Determine what to package
has_results = os.path.exists('results')
has_tuning = os.path.exists('results_tuning')

if has_tuning:
    # If tuning was run, package all tuning results
    print("📦 Packaging parameter tuning results...")
    !tar -czf results_tuning.tar.gz results_tuning/ 2>/dev/null
    
    if os.path.exists('results_tuning.tar.gz'):
        size_mb = os.path.getsize('results_tuning.tar.gz') / (1024 * 1024)
        print(f"✓ Archive created: results_tuning.tar.gz ({size_mb:.1f} MB)")
        print("\n⬇️ Downloading to your computer...")
        files.download('results_tuning.tar.gz')
        print("✓ Download complete!")
        print("\nTo extract on your Mac:")
        print("   tar -xzf results_tuning.tar.gz")
    else:
        print("⚠️ Failed to create tuning archive")

elif has_results:
    # If only single run results exist, package those
    print("📦 Packaging results...")
    !tar -czf results.tar.gz results/ 2>/dev/null
    
    if os.path.exists('results.tar.gz'):
        size_mb = os.path.getsize('results.tar.gz') / (1024 * 1024)
        print(f"✓ Archive created: results.tar.gz ({size_mb:.1f} MB)")
        print("\n⬇️ Downloading to your computer...")
        files.download('results.tar.gz')
        print("✓ Download complete!")
        print("\nTo extract on your Mac:")
        print("   tar -xzf results.tar.gz")
    else:
        print("⚠️ Failed to create results archive")

else:
    print("⚠️ No results to package. Make sure Step 5 or parameter tuning completed successfully.")

---

# Parameter Tuning

Run a ** grid search** to find optimal parameters with automatic quality metrics.

**What this does:**
- Tests 9 parameter combinations (instead of 36) covering the useful range
- Computes quality metrics for each: contrast, detail preservation, and color stability
- Ranks results by combined score to find the best parameters
- Much smaller output files (easier to review)

**Parameters tested:**
- iterations: [3, 5, 10] 
- sigma: [10, 15, 25] - controls smoothing 
- distance: [1.0] 

**Note**: Takes ~3-5 minutes for 3 images × 9 combinations = 27 runs

In [ ]:
import os
import sys
import shutil
import numpy as np
from PIL import Image
from datetime import datetime

%cd /content/Spectral_Ratio_Illumination_Demo

# Add current directory to Python path so imports work
sys.path.insert(0, '/content/Spectral_Ratio_Illumination_Demo')

# Define REDUCED parameter ranges to test (9 combinations instead of 36)
iterations_values = [3, 5, 10]  # Number of Retinex iterations
sigma_values = [10, 15, 25]     # Gaussian blur sigma (10=sharp, 15=balanced, 25=smooth)
distance_values = [1.0]         # SR color correction distance (fixed for simplicity)

print("🔬 Starting Smart Parameter Tuning")
print(f"   Testing {len(iterations_values)} × {len(sigma_values)} × {len(distance_values)} = {len(iterations_values) * len(sigma_values) * len(distance_values)} combinations")
print("   (Reduced from 36 to 9 for manageable file sizes)\n")

# Create a directory for tuning results
tuning_dir = 'results_tuning'
os.makedirs(tuning_dir, exist_ok=True)

total_runs = len(iterations_values) * len(sigma_values) * len(distance_values)
current_run = 0

# Quality metrics storage
quality_scores = []

def compute_quality_metrics(result_dir, param_str):
    """Compute simple quality metrics for ranking results."""
    try:
        # Find SR-Retinex output images
        sr_files = [f for f in os.listdir(result_dir) if 'sr_retinex_vis.png' in f]
        if not sr_files:
            return 0.0
        
        # Compute average metrics across all images
        total_contrast = 0
        total_detail = 0
        count = 0
        
        for img_file in sr_files:
            img_path = os.path.join(result_dir, img_file)
            img = np.array(Image.open(img_path).convert('L'))  # Grayscale
            
            # Contrast: standard deviation (higher = better contrast)
            contrast = np.std(img) / 128.0  # Normalize to ~[0, 2]
            
            # Detail preservation: high-frequency content (edges)
            from scipy import ndimage
            edges = ndimage.sobel(img)
            detail = np.mean(np.abs(edges)) / 50.0  # Normalize
            
            total_contrast += contrast
            total_detail += detail
            count += 1
        
        if count == 0:
            return 0.0
        
        avg_contrast = total_contrast / count
        avg_detail = total_detail / count
        
        # Combined score (weighted average)
        score = 0.5 * avg_contrast + 0.5 * avg_detail
        
        return score
    except Exception as e:
        print(f"   ⚠️ Could not compute metrics: {e}")
        return 0.0

# Test each combination
for iterations in iterations_values:
    for sigma in sigma_values:
        for distance in distance_values:
            current_run += 1
            param_str = f"iter{iterations}_sigma{sigma}_dist{distance}"
            print(f"\n{'='*70}")
            print(f"Run {current_run}/{total_runs}: {param_str}")
            print('='*70)
            
            # Run the batch processor with these parameters
            !python scripts/run_batch.py \
                --use-model \
                --retinex \
                --baseline-retinex \
                --sr-correct \
                --iterations {iterations} \
                --sigma {sigma} \
                --distance {distance}
            
            # Move results to a labeled subdirectory
            result_subdir = f"{tuning_dir}/{param_str}"
            if os.path.exists('results'):
                if os.path.exists(result_subdir):
                    shutil.rmtree(result_subdir)
                shutil.copytree('results', result_subdir)
                print(f"✓ Results saved to: {result_subdir}")
                
                # Compute quality score
                score = compute_quality_metrics(result_subdir, param_str)
                quality_scores.append((param_str, iterations, sigma, distance, score))
                print(f"   Quality score: {score:.3f}")
            
            # Clean up results directory for next run
            if os.path.exists('results'):
                shutil.rmtree('results')

print(f"\n\n{'='*70}")
print("✓ Parameter tuning complete!")
print(f"   All results saved in: {tuning_dir}/")
print('='*70)

# Rank by quality score
quality_scores.sort(key=lambda x: x[4], reverse=True)

print("\n📊 Parameter Ranking (by quality score):")
print(f"{'Rank':<6} {'Parameters':<30} {'Score':<8}")
print("-" * 50)
for rank, (param_str, iters, sig, dist, score) in enumerate(quality_scores, 1):
    print(f"{rank:<6} iter={iters:<2} sigma={sig:<3} dist={dist:<4} {score:>6.3f}")

if quality_scores:
    best_params = quality_scores[0]
    print(f"\n🏆 Best parameters: {best_params[0]} (score: {best_params[4]:.3f})")
    print(f"   Recommended for your final results!")

print("\n💡 Next steps:")
print("   1. Review top 3 parameter combinations visually")
print("   2. Use 'View Sample Results' cell below to compare")
print("   3. Run Step 7 to download all tuning results")

## View Top Parameter Results

After running parameter tuning above, use this cell to visually compare the top-ranked parameter combinations.

In [ ]:
import os
import matplotlib.pyplot as plt
from PIL import Image

tuning_dir = 'results_tuning'

if not os.path.exists(tuning_dir):
    print("⚠️ Run parameter tuning cell above first")
elif not quality_scores:
    print("⚠️ No quality scores available")
else:
    # Show top 3 results
    top_n = min(3, len(quality_scores))
    print(f"📊 Showing top {top_n} parameter combinations:\n")
    
    for rank in range(top_n):
        param_str, iters, sig, dist, score = quality_scores[rank]
        result_dir = os.path.join(tuning_dir, param_str)
        
        print(f"{rank+1}. {param_str} (score: {score:.3f})")
        
        # Find one image from this parameter set
        files = os.listdir(result_dir)
        base_names = set()
        for f in files:
            if '_image_8bit.png' in f:
                base_names.add(f.replace('_image_8bit.png', ''))
        
        if base_names:
            base_name = sorted(base_names)[0]
            
            # Show 4-way comparison
            fig, axes = plt.subplots(1, 4, figsize=(20, 5))
            
            images = [
                (f'{base_name}_image_8bit.png', 'Original'),
                (f'{base_name}_baseline_retinex_vis.png', 'Baseline'),
                (f'{base_name}_sr_retinex_vis.png', 'SR-Constrained'),
                (f'{base_name}_sr_shifted_vis.png', 'SR Corrected')
            ]
            
            for idx, (filename, label) in enumerate(images):
                img_path = os.path.join(result_dir, filename)
                if os.path.exists(img_path):
                    img = Image.open(img_path)
                    axes[idx].imshow(img)
                    axes[idx].set_title(label, fontsize=12, fontweight='bold')
                axes[idx].axis('off')
            
            plt.suptitle(f'Rank {rank+1}: {param_str} (score: {score:.3f})', 
                        fontsize=14, fontweight='bold')
            plt.tight_layout()
            plt.show()
            print()
    
    print("💡 Compare visual quality vs. quantitative scores to choose final parameters")

## View Sample Results (Step 5 outputs)

This displays results from **Step 5** (the main experiment run) for presentation.

**Shows 4 method comparison:**
- Original input (8-bit reference)
- Baseline Retinex (standard method - may shift colors)
- SR-constrained Retinex (your method - preserves colors better)
- SR color correction (simple illumination adjustment)

**Expected Results:**
- SR-constrained should preserve color better than baseline (less color shifts)
- Baseline may over-smooth or change colors unrealistically
- SR correction should brighten shadows while maintaining color relationships

In [ ]:
import os
import glob
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

results_dir = 'results'

if not os.path.exists(results_dir):
    print("⚠️ No results found. Run Step 5 first.")
else:
    # Find all result files
    all_files = sorted(os.listdir(results_dir))
    
    # Find a base image name (extract from 8-bit reference images)
    base_names = set()
    for f in all_files:
        if f.endswith('_image_8bit.png'):
            base_names.add(f.replace('_image_8bit.png', ''))
    
    if not base_names:
        print("⚠️ No processed images found in results/")
    else:
        # Pick first image for visualization
        base_name = sorted(base_names)[0]
        print(f"📸 Showing method comparison for: {base_name}\n")
        
        # Define the comparison sequence with correct file naming
        comparisons = [
            (f'{base_name}_image_8bit.png', 'Original Input (8-bit)'),
            (f'{base_name}_baseline_retinex_vis.png', 'Baseline Retinex'),
            (f'{base_name}_sr_retinex_vis.png', 'SR-Constrained Retinex\n(Your Method)'),
            (f'{base_name}_sr_shifted_vis.png', 'SR Color Correction')
        ]
        
        # Load images
        images_to_show = []
        labels_to_show = []
        
        for filename, label in comparisons:
            img_path = os.path.join(results_dir, filename)
            if os.path.exists(img_path):
                images_to_show.append(img_path)
                labels_to_show.append(label)
            else:
                print(f"   ⚠️ Missing: {filename}")
        
        if len(images_to_show) >= 2:
            # Display in 2x2 grid
            n_images = len(images_to_show)
            fig, axes = plt.subplots(2, 2, figsize=(16, 16))
            axes = axes.flatten()
            
            for idx, (img_path, label) in enumerate(zip(images_to_show, labels_to_show)):
                img = Image.open(img_path)
                axes[idx].imshow(img)
                axes[idx].set_title(label, fontsize=16, fontweight='bold', pad=10)
                axes[idx].axis('off')
            
            # Hide unused subplots
            for idx in range(n_images, 4):
                axes[idx].axis('off')
            
            plt.tight_layout()
            plt.show()
            
            print(f"\n✅ Comparison displayed for: {base_name}")
            print("\n💡 What to look for in your presentation:")
            print("   1. Color preservation: SR-constrained should maintain colors better than baseline")
            print("   2. Detail recovery: Check shadow regions - should reveal detail without artifacts")
            print("   3. Natural appearance: SR methods should look more realistic than baseline")
            print("   4. Brightness: All corrected images should be brighter but with preserved colors")
            
            # Show additional images if available
            if len(base_names) > 1:
                print(f"\n📌 {len(base_names)} images processed. Showing first one.")
                print(f"   Other images: {', '.join(sorted(base_names)[1:3])}")
        else:
            print("⚠️ Not enough output images found. Make sure Step 5 completed successfully.")
            print(f"   Found files: {images_to_show}")